In [9]:
%reload_ext autoreload
%autoreload 2
import numpy as np
from ee_ipl_uv.clustering import ClusterClouds

hoehe, breite = 500, 500
test_target_img = {
    "B2": np.random.rand(hoehe, breite).astype(np.float32),
    "B3": np.random.rand(hoehe, breite).astype(np.float32),
    "B4": np.random.rand(hoehe, breite).astype(np.float32)
}
for b in ["B1", "B5", "B6", "B7", "B9", "B10", "B11"]:
    test_target_img[b] = np.random.rand(hoehe, breite).astype(np.float32)

test_background_pred = {b: arr * 0.9 for b, arr in test_target_img.items()}

lokale_maske = ClusterClouds(test_target_img, test_background_pred, n_clusters=3)

print("Ergebnis-Typ:", type(lokale_maske))
print("Ergebnis-Shape:", lokale_maske.shape)

c:\Users\Linus\miniconda3\envs\ee\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Ergebnis-Typ: <class 'numpy.ndarray'>
Ergebnis-Shape: (500, 500)


In [1]:
%pip install pystac-client planetary-computer rioxarray shapely

import numpy as np
import pystac_client
import planetary_computer
import rioxarray
import matplotlib.pyplot as plt
import json
from shapely.geometry import shape

# 1. Paste the STAC Data Loader function we designed
def load_local_landsat_as_dict(bbox, datetime_range):
    """
    Searches Microsoft STAC catalog and loads Landsat-8 bands as a dictionary of NumPy arrays.
    """
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )

    # Search for Landsat 8 Level-2 (Surface Reflectance) data matching your ROI
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime=datetime_range,
        query={"eo:cloud_cover": {"lt": 20}}  # Filter under 20% clouds globally
    )
    
    items = list(search.item_collection())
    if len(items) == 0:
        raise FileNotFoundError(f"No scenes found for range: {datetime_range}")
        
    print(f"-> Found {len(items)} scenes for range {datetime_range}. Loading the first one...")
    selected_item = items[0] 
    
    # Map STAC asset tags to standard model band names
    stac_band_mapping = {
        "coastal": "B1", "blue": "B2", "green": "B3", "red": "B4",
        "nir08": "B5", "swir16": "B6", "swir22": "B7", 
        "cirrus": "B9", "lwir11": "B10", "lwir12": "B11"
    }
    
    image_dict = {}
    
    # Read each required band asset, clip to your bounding box, and transform to numpy
    for stac_name, bme_name in stac_band_mapping.items():
        if stac_name in selected_item.assets:
            url = selected_item.assets[stac_name].href
            # Open over network lazily, clip directly, and read values into memory
            with rioxarray.open_rasterio(url) as da:
                # KORREKTUR: Tell rioxarray that our bbox is in standard lat/lon (EPSG:4326).
                # It will automatically translate the numbers to the image's UTM meters!
                clipped = da.rio.clip_box(*bbox, crs="EPSG:4326")
                
                # Landsat-8 SR values scale factor is 0.0000275, offset -0.2
                array_data = clipped.values[0].astype(np.float32) * 0.0000275 - 0.2
                image_dict[bme_name] = np.clip(array_data, 0, 1)
                
    return image_dict

geojson_pfad = "Landkreis_Wuerzburg.geojson" 

with open(geojson_pfad, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

# 1. Parse the geometry locally using Shapely
if geojson_data.get("type") == "FeatureCollection":
    # Combine all features in the collection into a single geometry object
    geoms = [shape(feature["geometry"]) for feature in geojson_data["features"]]
    from shapely.ops import unary_union
    local_geometry = unary_union(geoms)
else:
    local_geometry = shape(geojson_data)

# 2. Extract the bounding box coordinates automatically: (minx, miny, maxx, maxy)
# This perfectly maps to STAC's expected [min_lon, min_lat, max_lon, max_lat]
wuerzburg_bbox = list(local_geometry.bounds)

print("Success! Extracted Bounding Box coordinates:")
print(wuerzburg_bbox)

print("Step 1: Loading target image...")
target_image = load_local_landsat_as_dict(wuerzburg_bbox, "2025-05-20/2025-06-15")

print("\nStep 2: Loading background image...")
# To make reproject_match easy, let's load the background using raw DataArrays first,
# or we can simply force numpy to clip/interpolate both matrices to the exact same size.
# Let's write a quick matrix aligner directly on the NumPy dictionaries:

def align_numpy_dicts(target_dict, background_dict):
    """
    Forces all arrays in background_dict to match the exact matrix shapes 
    of target_dict by slicing or padding them to the same (height, width).
    """
    aligned_background = {}
    
    # Target dimensions
    ref_band = list(target_dict.keys())[0]
    t_h, t_w = target_dict[ref_band].shape
    
    for band, arr in background_dict.items():
        b_h, b_w = arr.shape
        
        # Determine shared overlap limits or crop/pad
        new_arr = np.zeros((t_h, t_w), dtype=np.float32)
        
        # Calculate bounding limits to copy safely without out-of-bounds crashes
        min_h = min(t_h, b_h)
        min_w = min(t_w, b_w)
        
        # Inject the background data into the target-shaped template grid
        new_arr[:min_h, :min_w] = arr[:min_h, :min_w]
        
        # If background was smaller, fill remaining space with edge data or zero
        aligned_background[band] = new_arr
        
    return aligned_background

# Align background shape precisely to target shape
print("Aligning matrix dimensions...")
aligned_background = align_numpy_dicts(target_image, load_local_landsat_as_dict(wuerzburg_bbox, "2025-04-01/2025-05-15"))

# 3. RUNNING LOGIC ENTIRELY ON LOCAL CPU
print("\nStep 3: Triggering migrated non-GEE pipeline orchestrator...")
from ee_ipl_uv.multitemporal_cloud_masking import LocalCloudClusterScore

# Run your fully migrated, GEE-free function!
final_mask = LocalCloudClusterScore(target_image, aligned_background)

print("\nSuccess! Final local mask generated.")
print("Mask type:", type(final_mask))
print("Mask shape:", final_mask.shape)

# 4. PLOTTING RESULT DIRECTLY IN NOTEBOOK
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Target Image (RGB approximation)")
# Combine B4 (Red), B3 (Green), B2 (Blue) for a quick natural look
rgb = np.stack([target_image["B4"], target_image["B3"], target_image["B2"]], axis=-1)
plt.imshow(rgb * 3) # Boost brightness slightly for visualization

plt.subplot(1, 2, 2)
plt.title("Migrated Cloud Mask Output")
plt.imshow(final_mask, cmap="tab10")
plt.colorbar(ticks=[0, 1, 2], label="0=Clear, 1=Shadow, 2=Cloud")
plt.show()

Note: you may need to restart the kernel to use updated packages.
Success! Extracted Bounding Box coordinates:
[9.6254973, 49.4804787, 10.1893862, 49.9535198]
Step 1: Loading target image...
-> Found 2 scenes for range 2025-05-20/2025-06-15. Loading the first one...

Step 2: Loading background image...
Aligning matrix dimensions...
-> Found 16 scenes for range 2025-04-01/2025-05-15. Loading the first one...

Step 3: Triggering migrated non-GEE pipeline orchestrator...


ImportError: cannot import name 'normalization' from 'ee_ipl_uv' (C:\uni\Image_processing_seminar\sips\code\ee_ipl_uv_non_gee_migration\ee_ipl_uv\__init__.py)